<a href="https://colab.research.google.com/github/MLfinal/Walmart-Recruiting---Store-Sales-Forecasting/blob/dev/models/deep_learning/tft/model_experiment_TFT_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TFT experiment v3 — seasonal residual + external covariates

v1/v2 results on top 500 active Store-Dept series:
- seasonal naive WMAE: `4969.77`
- TFT v1 raw target WMAE: `6200.95`
- TFT v2 log target WMAE: `6524.68`

Both direct TFT approaches failed to beat seasonal naive. v3 changes the problem:
- compute a 52-week seasonal naive baseline for every row;
- train TFT to predict the residual correction:
  `ResidualSales = Weekly_Sales - SeasonalNaive52`;
- final prediction is:
  `Prediction = SeasonalNaive52 + PredictedResidual`;
- WMAE is still computed on original Weekly_Sales scale.

Why: Walmart has strong yearly seasonality. Instead of making TFT learn the full sales level, v3 asks TFT to learn only the correction around a strong seasonal baseline.


In [ ]:
%pip install -q "torch>=2.3,<3" "pytorch-forecasting>=1.2,<2" "lightning>=2.2,<3" "wandb>=0.19,<1" "pandas>=2.2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4" "scikit-learn>=1.4,<2"

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import wandb

try:
    from lightning.pytorch import Trainer, seed_everything
    from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
    from lightning.pytorch.loggers import WandbLogger
except Exception:
    from pytorch_lightning import Trainer, seed_everything
    from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
    from pytorch_lightning.loggers import WandbLogger

from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import MAE

pd.set_option('display.max_columns', 120)
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())

In [ ]:
SEED = 42
seed_everything(SEED, workers=True)

CONFIG = {
    "seed": SEED,
    "validation_weeks": 39,
    "encoder_weeks": 39,
    "top_n_series": 500,
    "holiday_weight": 5.0,
    "batch_size": 512,
    "max_epochs": 10,
    "max_time_minutes": 25,
    "patience": 3,
    "learning_rate": 1e-4,
    "hidden_size": 16,
    "attention_head_size": 2,
    "dropout": 0.10,
    "hidden_continuous_size": 8,
    "gradient_clip_val": 0.1,
    "num_workers": 0,
    "limit_train_batches": 50,
    "limit_val_batches": 8,
    "baseline_top300_wmae": 7801.898566783023,
    "v1_top500_wmae": 6200.954359119016,
    "v1_top500_seasonal_naive_wmae": 4969.768735310006,
    "v2_top500_wmae": 6524.679969182181,
    "wandb_project": "Walmart-Recruiting---Store-Sales-Forecasting",
    "wandb_entity": "kende23-n-a",
    "wandb_group": "tft-experiments",
    "run_name": "tft_v3_seasonal_residual_external_covariates",
    "artifact_name": "tft-v3-seasonal-residual-external-covariates",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

DATA_DIR = Path('/content/drive/MyDrive/walmart_competition_data')
OUTPUT_DIR = Path('/content/artifacts/tft_v3_seasonal_residual')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG

## Load data and keep top active series

We keep top 500 series to compare v2 with v1 on the same scale. If v2 cannot beat seasonal naive here, full-data training is not justified yet.

In [ ]:
train_raw = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['Date'])
test_raw = pd.read_csv(DATA_DIR / 'test.csv', parse_dates=['Date'])
features_raw = pd.read_csv(DATA_DIR / 'features.csv', parse_dates=['Date'])
stores_raw = pd.read_csv(DATA_DIR / 'stores.csv')

required_train = {'Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday'}
required_features = {'Store', 'Date', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment'}
required_stores = {'Store', 'Type', 'Size'}
missing = {
    'train': sorted(required_train.difference(train_raw.columns)),
    'features': sorted(required_features.difference(features_raw.columns)),
    'stores': sorted(required_stores.difference(stores_raw.columns)),
}
if any(missing.values()):
    raise ValueError(missing)

train_raw = train_raw.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
test_raw = test_raw.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
features_raw = features_raw.sort_values(['Store', 'Date']).reset_index(drop=True)
stores_raw = stores_raw.sort_values('Store').reset_index(drop=True)

if CONFIG['top_n_series'] is not None:
    top_series = (
        train_raw.groupby(['Store', 'Dept'], as_index=False)['Weekly_Sales']
        .sum()
        .sort_values('Weekly_Sales', ascending=False)
        .head(CONFIG['top_n_series'])[['Store', 'Dept']]
    )
    before_rows = len(train_raw)
    train_raw = train_raw.merge(top_series.assign(_keep=1), on=['Store', 'Dept'], how='inner').drop(columns='_keep')
    print({'top_n_series': CONFIG['top_n_series'], 'train_rows_before': before_rows, 'train_rows_after': len(train_raw)})

print('train', train_raw.shape, train_raw['Date'].min(), train_raw['Date'].max())
print('features', features_raw.shape, features_raw['Date'].min(), features_raw['Date'].max())
print('stores', stores_raw.shape)
display(train_raw.head())

## Merge covariates and create log target

In [ ]:
all_train_dates = pd.Index(sorted(train_raw['Date'].unique()), name='Date')
test_dates = pd.Index(sorted(test_raw['Date'].unique()), name='Date')
val_dates = all_train_dates[-CONFIG['validation_weeks']:]
fit_dates = all_train_dates[:-CONFIG['validation_weeks']]
split_pos = len(fit_dates)

if len(test_dates) != CONFIG['validation_weeks']:
    raise ValueError(f"Expected test horizon {CONFIG['validation_weeks']}, got {len(test_dates)}")
if len(fit_dates) < CONFIG['encoder_weeks'] + CONFIG['validation_weeks']:
    raise ValueError('Not enough fit history for encoder + decoder windows.')

date_to_idx = {date: idx for idx, date in enumerate(all_train_dates)}

COVARIATE_REALS = [
    'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'CPI', 'Unemployment', 'Size'
]
CALENDAR_REALS = ['time_idx', 'week_sin', 'week_cos', 'month_sin', 'month_cos']
KNOWN_REALS = CALENDAR_REALS + COVARIATE_REALS
TARGET_COL = 'ResidualSales'
SEASONAL_COL = 'SeasonalNaive52'

# 52-week seasonal lookup is available for every validation target row and for training windows whose target
# starts at least 52 weeks after the beginning of the series. Rows without 52-week history are kept with 0
# fallback but the dataset min_prediction_idx below starts at week 52 to avoid training mostly on fallback values.
def add_seasonal_naive_52(df: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values(['Store', 'Dept', 'Date']).copy()
    out['Weekly_Sales_Original'] = out['Weekly_Sales'].astype(float)
    out['Weekly_Sales_Clipped'] = out['Weekly_Sales_Original'].clip(lower=0.0)
    out[SEASONAL_COL] = out.groupby(['Store', 'Dept'])['Weekly_Sales_Clipped'].shift(52)
    out[SEASONAL_COL] = out[SEASONAL_COL].fillna(0.0)
    out[TARGET_COL] = (out['Weekly_Sales_Clipped'] - out[SEASONAL_COL]).astype(float)
    return out

def make_model_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = add_seasonal_naive_52(df)
    feat = features_raw.drop(columns=[c for c in ['IsHoliday'] if c in features_raw.columns])
    out = out.merge(feat, on=['Store', 'Date'], how='left')
    out = out.merge(stores_raw, on='Store', how='left')

    markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
    out[markdown_cols] = out[markdown_cols].fillna(0.0)
    for col in ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']:
        out[col] = out.groupby('Store')[col].transform(lambda s: s.ffill().bfill())
        out[col] = out[col].fillna(out[col].median())
    out['Size'] = out['Size'].fillna(out['Size'].median())
    out['Type'] = out['Type'].fillna('Unknown')

    out['time_idx'] = out['Date'].map(date_to_idx).astype('int64')
    iso_week = out['Date'].dt.isocalendar().week.astype(int)
    month = out['Date'].dt.month.astype(int)
    out['week_sin'] = np.sin(2 * np.pi * iso_week / 52.0)
    out['week_cos'] = np.cos(2 * np.pi * iso_week / 52.0)
    out['month_sin'] = np.sin(2 * np.pi * month / 12.0)
    out['month_cos'] = np.cos(2 * np.pi * month / 12.0)

    out['Store'] = out['Store'].astype(str)
    out['Dept'] = out['Dept'].astype(str)
    out['Type'] = out['Type'].astype(str)
    out['IsHoliday'] = out['IsHoliday'].astype(int).astype(str)
    for col in KNOWN_REALS:
        if col != 'time_idx':
            out[col] = out[col].astype(float)
    out['time_idx'] = out['time_idx'].astype('int64')
    return out

data = make_model_frame(train_raw)

print({
    'n_rows': len(data),
    'n_series': data[['Store', 'Dept']].drop_duplicates().shape[0],
    'n_dates': len(all_train_dates),
    'fit_range': (str(fit_dates.min().date()), str(fit_dates.max().date())),
    'validation_range': (str(val_dates.min().date()), str(val_dates.max().date())),
    'target': TARGET_COL,
    'time_idx_dtype': str(data['time_idx'].dtype),
})
display(data[['Store', 'Dept', 'Date', 'Weekly_Sales_Original', 'Weekly_Sales_Clipped', SEASONAL_COL, TARGET_COL, 'IsHoliday'] + KNOWN_REALS[:5]].head())

## WMAE and seasonal naive on original sales scale

In [ ]:
def wmae(y_true: np.ndarray, y_pred: np.ndarray, is_holiday: np.ndarray, holiday_weight: float = 5.0) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    weights = np.where(np.asarray(is_holiday, dtype=bool), holiday_weight, 1.0)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))

sales_panel = (
    train_raw.pivot_table(index=['Store', 'Dept'], columns='Date', values='Weekly_Sales', aggfunc='sum')
    .reindex(columns=all_train_dates)
    .fillna(0.0)
    .sort_index()
)
holiday_by_date = (
    train_raw[['Date', 'IsHoliday']]
    .drop_duplicates('Date')
    .set_index('Date')
    .reindex(all_train_dates)['IsHoliday']
    .fillna(False)
    .astype(bool)
)
values = sales_panel.to_numpy(dtype=np.float32)
holiday_flags = holiday_by_date.to_numpy(dtype=bool)
actual_val = values[:, split_pos:split_pos + CONFIG['validation_weeks']]
seasonal_naive = values[:, split_pos - 52:split_pos - 52 + CONFIG['validation_weeks']]
val_holidays_matrix = np.tile(holiday_flags[split_pos:split_pos + CONFIG['validation_weeks']], (values.shape[0], 1))
seasonal_naive_wmae = wmae(actual_val.ravel(), seasonal_naive.ravel(), val_holidays_matrix.ravel(), CONFIG['holiday_weight'])
print({'n_series': len(sales_panel), 'seasonal_naive_wmae': seasonal_naive_wmae})

## Create TFT datasets/loaders

In [ ]:
training_cutoff = split_pos - 1

training_dataset = TimeSeriesDataSet(
    data[data.time_idx <= training_cutoff],
    time_idx='time_idx',
    min_prediction_idx=52,
    target=TARGET_COL,
    group_ids=['Store', 'Dept'],
    min_encoder_length=CONFIG['encoder_weeks'] // 2,
    max_encoder_length=CONFIG['encoder_weeks'],
    min_prediction_length=CONFIG['validation_weeks'],
    max_prediction_length=CONFIG['validation_weeks'],
    static_categoricals=['Store', 'Dept', 'Type'],
    time_varying_known_categoricals=['IsHoliday'],
    time_varying_known_reals=KNOWN_REALS,
    time_varying_unknown_reals=[TARGET_COL],
    target_normalizer=GroupNormalizer(groups=['Store', 'Dept'], center=True),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)
validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    data,
    predict=True,
    stop_randomization=True,
    min_prediction_idx=split_pos,
)
train_loader = training_dataset.to_dataloader(train=True, batch_size=CONFIG['batch_size'], num_workers=CONFIG['num_workers'])
val_loader = validation_dataset.to_dataloader(train=False, batch_size=CONFIG['batch_size'], num_workers=CONFIG['num_workers'])
print({
    'train_batches_total': len(train_loader),
    'validation_batches_total': len(val_loader),
    'training_samples': len(training_dataset),
    'validation_samples': len(validation_dataset),
    'effective_train_batches_per_epoch': min(len(train_loader), CONFIG['limit_train_batches']),
})

## W&B setup

In [ ]:
baseline_model = Baseline()
_ = baseline_model.predict(val_loader, return_y=True)
print('PyTorch Forecasting Baseline object ran successfully.')

if wandb.run is not None:
    wandb.finish()

wandb_logger = WandbLogger(
    project=CONFIG['wandb_project'],
    entity=CONFIG['wandb_entity'],
    group=CONFIG['wandb_group'],
    name=CONFIG['run_name'],
    job_type='experiment_train',
    log_model=False,
    config=CONFIG,
)
run = wandb_logger.experiment
run.config.update({
    'n_series': int(data[['Store', 'Dept']].drop_duplicates().shape[0]),
    'n_train_rows': int(len(data[data.time_idx <= training_cutoff])),
    'n_validation_rows': int(len(data[data.time_idx >= split_pos])),
    'fit_start': str(fit_dates.min().date()),
    'fit_end': str(fit_dates.max().date()),
    'validation_start': str(val_dates.min().date()),
    'validation_end': str(val_dates.max().date()),
    'seasonal_naive_wmae': float(seasonal_naive_wmae),
    'known_reals': json.dumps(KNOWN_REALS),
    'target_strategy': 'seasonal_residual_52w',
}, allow_val_change=True)

## Build and train TFT v2

In [ ]:
checkpoint_callback = ModelCheckpoint(
    dirpath=str(OUTPUT_DIR / 'checkpoints'),
    filename='tft-v3-{epoch:02d}-{val_loss:.4f}',
    monitor='val_loss',
    mode='min',
    save_top_k=1,
)
early_stop_callback = EarlyStopping(monitor='val_loss', min_delta=1e-4, patience=CONFIG['patience'], mode='min')
lr_monitor = LearningRateMonitor(logging_interval='epoch')

tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=CONFIG['learning_rate'],
    hidden_size=CONFIG['hidden_size'],
    attention_head_size=CONFIG['attention_head_size'],
    dropout=CONFIG['dropout'],
    hidden_continuous_size=CONFIG['hidden_continuous_size'],
    loss=MAE(),
    optimizer='adam',
    log_interval=20,
    reduce_on_plateau_patience=3,
)
print(f'Number of parameters: {tft.size() / 1e3:.1f}k')
run.summary['model_parameters'] = int(tft.size())

trainer = Trainer(
    max_epochs=CONFIG['max_epochs'],
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    gradient_clip_val=CONFIG['gradient_clip_val'],
    callbacks=[early_stop_callback, checkpoint_callback, lr_monitor],
    logger=wandb_logger,
    enable_checkpointing=True,
    limit_train_batches=CONFIG['limit_train_batches'],
    limit_val_batches=CONFIG['limit_val_batches'],
    max_time={"minutes": CONFIG['max_time_minutes']},
    log_every_n_steps=20,
)
trainer.fit(tft, train_dataloaders=train_loader, val_dataloaders=val_loader)

best_checkpoint_path = checkpoint_callback.best_model_path
best_val_loss = float(checkpoint_callback.best_model_score.cpu()) if checkpoint_callback.best_model_score is not None else math.nan
print({'best_checkpoint_path': best_checkpoint_path, 'best_val_loss': best_val_loss})
run.summary['best_checkpoint_path'] = best_checkpoint_path
run.summary['best_val_loss'] = best_val_loss

## Evaluate validation WMAE with seasonal residual reconstruction

The model predicts residuals. We reconstruct final sales prediction as:

```text
Prediction = SeasonalNaive52 + PredictedResidual
Prediction = clip(Prediction, lower=0)
```


In [ ]:
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_checkpoint_path)
best_tft.eval()

prediction = best_tft.predict(
    val_loader,
    mode='prediction',
    return_index=True,
    return_y=True,
    trainer_kwargs={'accelerator': 'gpu' if torch.cuda.is_available() else 'cpu', 'devices': 1},
)
pred_residual = prediction.output.detach().cpu().numpy() if torch.is_tensor(prediction.output) else np.asarray(prediction.output)
index_df = prediction.index.reset_index(drop=True).copy()
if pred_residual.ndim == 3:
    pred_residual = pred_residual[..., 0]
if pred_residual.shape[1] != CONFIG['validation_weeks']:
    raise ValueError(f"Expected prediction horizon {CONFIG['validation_weeks']}, got {pred_residual.shape}")

actual_lookup = sales_panel.copy()
seasonal_lookup = data.pivot_table(index=['Store', 'Dept'], columns='Date', values=SEASONAL_COL, aggfunc='first').reindex(index=sales_panel.index, columns=all_train_dates).fillna(0.0)
records = []
for row_idx, row in index_df.iterrows():
    store = int(row['Store'])
    dept = int(row['Dept'])
    if (store, dept) not in actual_lookup.index:
        continue
    for horizon_idx, date in enumerate(val_dates):
        actual = float(actual_lookup.loc[(store, dept), date])
        seasonal_base = float(seasonal_lookup.loc[(store, dept), date])
        residual_pred = float(pred_residual[row_idx, horizon_idx])
        pred = float(max(seasonal_base + residual_pred, 0.0))
        is_holiday = bool(holiday_by_date.loc[date])
        records.append({
            'Store': store,
            'Dept': dept,
            'Date': pd.Timestamp(date),
            'IsHoliday': is_holiday,
            'Weekly_Sales': actual,
            'SeasonalNaive52': seasonal_base,
            'PredictedResidual': residual_pred,
            'Prediction': pred,
            'AbsError': abs(actual - pred),
        })
val_pred_df = pd.DataFrame(records)
if val_pred_df.empty:
    raise ValueError('No validation predictions were aligned.')

validation_wmae = wmae(val_pred_df['Weekly_Sales'], val_pred_df['Prediction'], val_pred_df['IsHoliday'], CONFIG['holiday_weight'])
improvement_vs_seasonal = 100.0 * (seasonal_naive_wmae - validation_wmae) / seasonal_naive_wmae
improvement_vs_v1 = 100.0 * (CONFIG['v1_top500_wmae'] - validation_wmae) / CONFIG['v1_top500_wmae']
improvement_vs_v2 = 100.0 * (CONFIG['v2_top500_wmae'] - validation_wmae) / CONFIG['v2_top500_wmae']
print({
    'validation_wmae': validation_wmae,
    'seasonal_naive_wmae': seasonal_naive_wmae,
    'improvement_vs_seasonal_naive_pct': improvement_vs_seasonal,
    'improvement_vs_v1_pct': improvement_vs_v1,
    'improvement_vs_v2_pct': improvement_vs_v2,
    'prediction_rows': len(val_pred_df),
})
run.summary['best_validation_wmae'] = float(validation_wmae)
run.summary['seasonal_naive_wmae'] = float(seasonal_naive_wmae)
run.summary['best_improvement_vs_seasonal_naive_pct'] = float(improvement_vs_seasonal)
run.summary['improvement_vs_v1_pct'] = float(improvement_vs_v1)
run.summary['improvement_vs_v2_pct'] = float(improvement_vs_v2)
wandb.log({
    'validation/wmae': float(validation_wmae),
    'validation/seasonal_naive_wmae': float(seasonal_naive_wmae),
    'validation/improvement_vs_seasonal_naive_pct': float(improvement_vs_seasonal),
    'validation/improvement_vs_v1_pct': float(improvement_vs_v1),
    'validation/improvement_vs_v2_pct': float(improvement_vs_v2),
})
display(val_pred_df.head())

## Save artifacts to W&B

In [ ]:
val_pred_path = OUTPUT_DIR / 'tft_v3_validation_predictions.csv'
val_pred_df.to_csv(val_pred_path, index=False)
summary = {
    'model': 'TemporalFusionTransformer',
    'experiment': 'tft_v3_seasonal_residual_external_covariates',
    'target_strategy': 'seasonal_residual_52w',
    'best_checkpoint_path': str(best_checkpoint_path),
    'best_val_loss': best_val_loss,
    'best_validation_wmae': float(validation_wmae),
    'seasonal_naive_wmae': float(seasonal_naive_wmae),
    'improvement_vs_seasonal_naive_pct': float(improvement_vs_seasonal),
    'improvement_vs_v1_pct': float(improvement_vs_v1),
    'improvement_vs_v2_pct': float(improvement_vs_v2),
    'validation_weeks': CONFIG['validation_weeks'],
    'encoder_weeks': CONFIG['encoder_weeks'],
    'top_n_series': CONFIG['top_n_series'],
    'n_series': int(data[['Store', 'Dept']].drop_duplicates().shape[0]),
    'known_reals': KNOWN_REALS,
    'config': CONFIG,
}
summary_path = OUTPUT_DIR / 'tft_v3_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))

fig, ax = plt.subplots(figsize=(7,4))
sample = val_pred_df.sample(min(7000, len(val_pred_df)), random_state=SEED)
ax.scatter(sample['Weekly_Sales'], sample['Prediction'], s=8, alpha=0.25)
max_axis = np.nanpercentile(sample[['Weekly_Sales', 'Prediction']].to_numpy(), 99)
ax.plot([0, max_axis], [0, max_axis], color='red', linewidth=1)
ax.set_title('TFT v3 seasonal residual validation predictions')
ax.set_xlabel('Actual Weekly_Sales')
ax.set_ylabel('Prediction')
plt.tight_layout()
scatter_path = OUTPUT_DIR / 'tft_v3_validation_scatter.png'
fig.savefig(scatter_path, dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(8,4))
ax.hist(val_pred_df['AbsError'], bins=80)
ax.set_title('TFT v3 validation absolute error distribution')
ax.set_xlabel('Absolute error')
ax.set_ylabel('count')
plt.tight_layout()
error_hist_path = OUTPUT_DIR / 'tft_v3_abs_error_hist.png'
fig.savefig(error_hist_path, dpi=160)
plt.show()

artifact = wandb.Artifact(CONFIG['artifact_name'], type='model')
artifact.add_file(str(best_checkpoint_path), name='tft_v3_best.ckpt')
artifact.add_file(str(summary_path))
artifact.add_file(str(val_pred_path))
artifact.add_file(str(scatter_path))
artifact.add_file(str(error_hist_path))
run.log_artifact(artifact, aliases=['experiment-v3', 'latest'])
wandb.log({
    'validation/prediction_table': wandb.Table(dataframe=val_pred_df.sample(min(20000, len(val_pred_df)), random_state=SEED)),
    'validation/scatter': wandb.Image(str(scatter_path)),
    'validation/abs_error_histogram': wandb.Image(str(error_hist_path)),
})
summary

In [ ]:
wandb.finish()